### Setup

In [1]:
import numpy as np
from scipy.stats import norm

alpha = 0.95
S0 = 152.51
K = 170.0
T = 0.292
r = 0.00119
sigma = 0.4907
mu = 0.1691
step = 1/252
days = 10
puts = 100
trials = 40000

def put_price(time, spot):
    tau = T - time
    d1 = (np.log(spot/K) + (r + 0.5*sigma**2)*tau)/(sigma*np.sqrt(tau))
    d2 = d1 - sigma*np.sqrt(tau)
    return K*np.exp(-r*tau)*norm.cdf(-d2) - spot*norm.cdf(-d1)

def put_delta(time, spot):
    tau = T - time
    d1 = (np.log(spot/K) + (r + 0.5*sigma**2)*tau)/(sigma*np.sqrt(tau))
    return norm.cdf(d1) - 1

def empirical_var(losses, level):
    ordered = np.sort(losses)
    return ordered[int(np.ceil(len(losses)*level)) - 1]

generator = np.random.default_rng(42)
returns = generator.normal((mu - 0.5*sigma**2)*step, sigma*np.sqrt(step), (trials, days))
prices = np.column_stack([np.full(trials, S0), S0*np.exp(np.cumsum(returns, axis=1))])
print("put price", round(put_price(0, S0), 2))
print("delta", round(put_delta(0, S0), 4))

put price 27.11
delta -0.6086


### 1.

In [2]:
hedge = put_delta(0, S0)
money = -hedge*S0
value_start = money + hedge*S0 - put_price(0, S0)
spot = prices[:, 1]
value = hedge*spot + money*np.exp(r*step) - put_price(step, spot)
losses_one_day = -puts*(value - value_start)
var_one_day = empirical_var(losses_one_day, alpha)
print("one day VaR", round(var_one_day, 2))
print("ten day VaR by square root of time", round(np.sqrt(days)*var_one_day, 2))

one day VaR 29.93
ten day VaR by square root of time 94.64


### 2.

In [3]:
hedge = put_delta(0, S0)
money = -hedge*S0
for j in range(1, days):
    money = money*np.exp(r*step)
    spot = prices[:, j]
    new_hedge = put_delta(j*step, spot)
    money = money + (hedge - new_hedge)*spot
    hedge = new_hedge

spot = prices[:, days]
value = hedge*spot + money*np.exp(r*step) - put_price(days*step, spot)
losses_ten_day = -puts*(value - value_start)
var_ten_day = empirical_var(losses_ten_day, alpha)
print("ten day VaR by time aggregation", round(var_ten_day, 2))

ten day VaR by time aggregation 89.29


### 3.

In [4]:
from scipy.stats import skew
print("one day loss, standard deviation", round(losses_one_day.std(ddof=1), 2),"skewness", round(skew(losses_one_day), 2))
print("ten day loss, standard deviation", round(losses_ten_day.std(ddof=1), 2),"skewness", round(skew(losses_ten_day), 2))
print("square root of ten times the one day standard deviation", round(np.sqrt(days)*losses_one_day.std(ddof=1), 2))

one day loss, standard deviation 14.94 skewness 2.82
ten day loss, standard deviation 48.48 skewness 0.96
square root of ten times the one day standard deviation 47.24


Scaling the one day VaR by the square root of ten gives 94.64 dollars and running the hedge forward
for ten days gives 89.29 dollars, so the scaling rule comes out about six percent high. The gap does
not come from the position being riskier or safer over the longer window, which was the first thing I
checked, because the spread of the loss actually grows slightly faster than the square root of time,
from 14.94 dollars in a day to 48.48 dollars over ten where scaling would have predicted 47.24. What
moves is the shape of the distribution rather than its width. A single day loss on a short put that
is only delta hedged leans hard to the right, with skewness near 2.8, since a large move in either
direction hurts a position that is short gamma while a calm day hands back no more than the small
theta amount. Over the ten days the hedge gets reset nine times, so the total ends up behaving much
like a sum of ten separate daily losses and the usual averaging effect pulls it towards a bell shape
with skewness down around 1.0. Because the 95 percent point sits far deeper in the tail of the skewed
daily distribution than it does in the tamer ten day one, multiplying by the square root of ten
carries that stretched tail along with it.

Thinking about when the rule would have been exact makes the reason clearer to me. Square root of
time scaling needs the daily losses to be independent with the same distribution, and it needs that
distribution to keep its shape when copies of it are added together, which in practice means a normal
with zero mean. Neither condition holds here, since both the delta and the time left to maturity move
as the simulation runs so the daily losses are not identically distributed, and the curvature of the
option keeps each one well away from normal. I would trust the time aggregation figure over the
scaled one because it simulates the hedge the account actually runs rather than assuming the daily
risk simply stacks up.